# Vectors and similarity · Module 02
Python 3 and NumPy only. Predict every printed number before you run its cell, then explain any surprise. The final experiment asks you to change one thing.

Synthetic bars teach mechanics. They are not evidence about real photographs.


## Part 1: Dot products, angles and projection
Work these by hand first (Lessons 05 and 06). The residual is what the projection leaves behind, so it must be perpendicular to `u`.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(202)
x = np.array([1., 2.]); w = np.array([3., -1.])
print('dot product w.x:', w @ x)
assert w @ x == 1
print('length of x:', np.linalg.norm(x))
print('cosine of the angle:', (w @ x) / (np.linalg.norm(w) * np.linalg.norm(x)))
u = np.array([1., 1.]); y = np.array([3., 4.])
projection = u * (u @ y) / (u @ u)
residual = y - projection
assert np.isclose(u @ residual, 0)
print('projection:', projection, 'residual:', residual)


dot product w.x: 1.0
length of x: 2.23606797749979
cosine of the angle: 0.14142135623730948
projection: [3.5 3.5] residual: [-0.5  0.5]


## Part 2: A layer is a matrix product plus a bias
Examples are stored as rows, so `X` has shape (N, D). Write the output shape before running the cell. Then notice that `*` and `@` are different operations and that matrix order matters.


In [2]:
X = np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.],[1.,1.,1.]])   # (N, D) = (4, 3)
W = np.array([[2.,0.],[0.,1.],[1.,1.]])                        # (D, H) = (3, 2)
b = np.array([.5, -.5])                                        # one bias per output
Y = X @ W + b
print('X', X.shape, '@ W', W.shape, '+ b', b.shape, 'gives', Y.shape)
print(Y)
assert Y.shape == (4, 2)
print('trainable numbers in this layer:', W.size + b.size)
A = np.array([[1.,2.],[3.,4.]]); B = np.array([[0.,1.],[1.,0.]])
print('elementwise A*B:\n', A * B)
print('matrix product A@B:\n', A @ B)
print('matrix product B@A:\n', B @ A)
assert not np.allclose(A @ B, B @ A)


X (4, 3) @ W (3, 2) + b (2,) gives (4, 2)
[[ 2.5 -0.5]
 [ 0.5  0.5]
 [ 1.5  0.5]
 [ 3.5  1.5]]
trainable numbers in this layer: 8
elementwise A*B:
 [[0. 2.]
 [3. 0.]]
matrix product A@B:
 [[2. 1.]
 [4. 3.]]
matrix product B@A:
 [[3. 4.]
 [1. 2.]]


## Part 3: Rank and information lost
`B` below squeezes the plane onto a line. Find a nonzero vector it sends to zero, then see that adding it to any input changes nothing in the output. That lost direction is the null space.


In [3]:
B = np.array([[1.,1.],[2.,2.]])
null = np.array([1.,-1.])
assert np.allclose(B @ null, 0)
print('rank of B:', np.linalg.matrix_rank(B))
x = np.array([3.,5.])
print('B@x:', B @ x, 'and B@(x + 2*null):', B @ (x + 2 * null))
assert np.allclose(B @ x, B @ (x + 2 * null))
print('rank of diag(2,1):', np.linalg.matrix_rank(np.diag([2.,1.])))


rank of B: 1
B@x: [ 8. 16.] and B@(x + 2*null): [ 8. 16.]
rank of diag(2,1): 2


## Part 4: Nearest-centroid recognition on 8x8 bars
Each image is flattened to a 64-dimensional vector. The classifier stores one average vector per class, fitted on training data only, and compares a new image with both. Compare Euclidean distance with normalized dot-product (cosine) similarity.

Expand the squared distance: ||x - c||² = ||x||² - 2 x·c + ||c||². When both centroids have the same length, the smallest distance and the largest dot product pick the same class, so expect the two rules to agree on this first dataset.


In [4]:
def bars(n, noise=.12, levels=(1., 1.)):
    y = rng.integers(0, 2, n); images = np.zeros((n, 8, 8))
    for i, label in enumerate(y):
        p = int(rng.integers(1, 6))
        if label == 0: images[i, p:p+2, :] = levels[0]
        else:          images[i, :, p:p+2] = levels[1]
    return (images + rng.normal(0, noise, images.shape)).reshape(n, -1), y
def show(row):
    shades = ' .:-=+*#%@'
    for line in np.clip(row.reshape(8, 8), 0, .99):
        print(''.join(shades[int(v * 10)] * 2 for v in line))
X, y = bars(240)
Xtr, ytr, Xval, yval = X[:160], y[:160], X[160:], y[160:]
print('one training example, label', ytr[0]); show(Xtr[0])
centers = np.stack([Xtr[ytr == c].mean(axis=0) for c in [0, 1]])
print('centroids', centers.shape, '(classes, pixels)')
def euclidean(new, centers):
    return np.sum((new[:, None, :] - centers[None, :, :]) ** 2, axis=2).argmin(axis=1)
def cosine(new, centers):
    a = new / np.linalg.norm(new, axis=1, keepdims=True)
    b = centers / np.linalg.norm(centers, axis=1, keepdims=True)
    return (a @ b.T).argmax(axis=1)
print('validation accuracy, Euclidean:', np.mean(euclidean(Xval, centers) == yval))
print('validation accuracy, cosine:   ', np.mean(cosine(Xval, centers) == yval))
assert np.mean(cosine(Xval, centers) == yval) > .9
assert np.mean(euclidean(Xval, centers) == cosine(Xval, centers)) > .9


one training example, label 0
    --          
......    --    
                
@@@@**@@##@@@@@@
@@@@@@@@@@@@@@**
                
        ..  ::  
              ::
centroids (2, 64) (classes, pixels)
validation accuracy, Euclidean: 0.9125
validation accuracy, cosine:    0.9125


## Part 5: When length misleads
Suppose the training photos of class 1 happened to be taken in brighter light. Brightness now changes the length of a centroid without saying anything about the label. Predict which rule suffers on evenly lit test images, using the expansion from Part 4.


In [5]:
Xlit, ylit = bars(160, levels=(.6, 1.8))
lit = np.stack([Xlit[ylit == c].mean(axis=0) for c in [0, 1]])
print('centroid lengths:', np.linalg.norm(lit, axis=1))
results = {}
for level in [.5, 1.]:
    Xtest, ytest = bars(400, levels=(level, level))
    results[level] = np.mean(euclidean(Xtest, lit) == ytest), np.mean(cosine(Xtest, lit) == ytest)
    print(f'test images at brightness {level}: Euclidean {results[level][0]:.3f}, cosine {results[level][1]:.3f}, N=400')
assert results[.5][1] > results[.5][0] + .2


centroid lengths: [1.5119 4.5226]
test images at brightness 0.5: Euclidean 0.505, cosine 0.853, N=400
test images at brightness 1.0: Euclidean 0.672, cosine 0.860, N=400


## Your experiment
Write the shape of every array in Part 4 on paper: this is your shape-annotated forward pass. Explain the result of Part 5 with the geometry of length and angle. Then change one thing, such as noise or bar width, and report both accuracies with their counts. Fit centroids on training data only, and say what your result does not show about real photographs.
